# P7-1_Evaluacion_Desbalance_Clases

Evaluacion del efecto del desbalance de clases en el protocolo temporal corregido.

Escenarios: 1:1, 1:5 y 1:10 (positivo:negativos).
- threshold de coautoria fijo = 1.
- usa el Top-7 congelado por P5_scores_features.
- combina train + validation de 2021-2022 como desarrollo.
- selecciona hiperparametros con RepeatedStratifiedKFold 5x20 dentro de desarrollo.
- imputacion y escalado ocurren dentro del pipeline de CV.
- el test 2023-2024 no participa en seleccion ni ajuste.
- AUC-PR es la metrica principal bajo desbalance.
- ejecutar primero con RUN_FINAL_TEST = False.

In [2]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score, recall_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings('ignore')

SEED = 41
N_SPLITS = 5
N_REPEATS = 20
RUN_FINAL_TEST = True

BASE_DIR = Path('C:/Users/Dra. Pilar/UltimaversionProgramasComparacion')
P7_DIR = BASE_DIR / 'P7_desbalance_clases'
OUTPUT_DIR = P7_DIR

SCENARIOS = {
    '1:1': 1,
    '1:5': 5,
    '1:10': 10,
}

TOP7 = [
    'sum_of_neighbors',
    'sum_of_papers',
    'sum_of_keywords',
    'community_cn',
    'keywords_match',
    'adamic_adar_index',
    'resource_allocation',
]
TARGET = 'connected'


def files_for_ratio(ratio):
    suffix = f'thr1_neg{ratio}_random_isolates'
    return (
        P7_DIR / f'sample_temporal_train_hist2000_2020_target2021_2022_{suffix}.csv',
        P7_DIR / f'sample_temporal_val_hist2000_2020_target2021_2022_{suffix}.csv',
        P7_DIR / f'sample_temporal_test_hist2000_2022_target2023_2024_{suffix}.csv',
    )


def decision_score(model, X):
    if hasattr(model, 'predict_proba'):
        return np.asarray(model.predict_proba(X))[:, 1]
    if hasattr(model, 'decision_function'):
        return np.asarray(model.decision_function(X)).reshape(-1)
    return np.asarray(model.predict(X)).reshape(-1)


def make_model_spaces():
    return {
        'Decision Tree': (
            DecisionTreeClassifier(random_state=SEED),
            {
                'model__criterion': ['gini', 'entropy'],
                'model__max_depth': [2, 4, None],
                'model__min_samples_leaf': [1, 2, 4],
            }
        ),
        'SVM - Linear Kernel': (
            LinearSVC(random_state=SEED),
            {
                'model__C': [0.01, 0.1, 1.0, 10.0],
                'model__dual': [False],
            }
        ),
        'SVM - RBF Kernel': (
            SVC(kernel='rbf', probability=True, random_state=SEED),
            {
                'model__C': [0.1, 1.0, 5.0, 10.0],
                'model__gamma': ['scale', 'auto'],
            }
        ),
        'K-Neighbors': (
            KNeighborsClassifier(),
            {
                'model__n_neighbors': [3, 5, 7, 11],
                'model__weights': ['uniform', 'distance'],
                'model__p': [1, 2],
            }
        ),
        'Naive Bayes': (
            GaussianNB(),
            {
                'model__var_smoothing': [1e-11, 1e-9, 1e-7, 1e-5],
            }
        ),
        'Multi-Layer Perceptron': (
            MLPClassifier(max_iter=1000, random_state=SEED),
            {
                'model__hidden_layer_sizes': [(25,), (50,)],
                'model__alpha': [0.0001, 0.001],
                'model__learning_rate_init': [0.001, 0.01],
            }
        ),
    }


cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

all_searches = {}
all_data = {}
cv_rows = []
best_params_all = {}
audit_rows = []

for scenario, ratio in SCENARIOS.items():
    print('\n' + '=' * 78)
    print(f'ESCENARIO {scenario}  (1 positivo : {ratio} negativos)')
    print('=' * 78)

    train_file, val_file, test_file = files_for_ratio(ratio)
    for path in (train_file, val_file, test_file):
        if not path.exists():
            raise FileNotFoundError(f'No se encontro: {path}')

    train_df = pd.read_csv(train_file)
    val_df = pd.read_csv(val_file)
    test_df = pd.read_csv(test_file)

    for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        missing = [c for c in TOP7 + [TARGET] if c not in df.columns]
        if missing:
            raise ValueError(f'{scenario} {name}: faltan columnas {missing}')

    dev_df = pd.concat([train_df, val_df], ignore_index=True)
    X_dev = dev_df[TOP7].copy()
    y_dev = dev_df[TARGET].astype(int).to_numpy()
    X_test = test_df[TOP7].copy()
    y_test = test_df[TARGET].astype(int).to_numpy()

    dev_counts = pd.Series(y_dev).value_counts().sort_index().to_dict()
    test_counts = pd.Series(y_test).value_counts().sort_index().to_dict()
    prevalence_dev = float(np.mean(y_dev))
    prevalence_test = float(np.mean(y_test))

    print(f'train: n={len(train_df)}, clases={train_df[TARGET].value_counts().sort_index().to_dict()}')
    print(f'val: n={len(val_df)}, clases={val_df[TARGET].value_counts().sort_index().to_dict()}')
    print(f'desarrollo: n={len(dev_df)}, clases={dev_counts}, prevalencia positiva={prevalence_dev:.4f}')
    print(f'test reservado: n={len(test_df)}, clases={test_counts}, prevalencia positiva={prevalence_test:.4f}')

    searches = {}
    best_params_scenario = {}

    for model_name, (estimator, grid) in make_model_spaces().items():
        print(f'  Buscando hiperparametros: {model_name}')

        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', estimator),
        ])

        search = GridSearchCV(
            estimator=pipe,
            param_grid=grid,
            scoring={
                'auc_pr': 'average_precision',
                'auc_roc': 'roc_auc',
                'f1': 'f1',
                'accuracy': 'accuracy',
                'balanced_accuracy': 'balanced_accuracy',
                'precision': 'precision',
                'recall': 'recall',
            },
            refit='auc_pr',
            cv=cv,
            n_jobs=-1,
            return_train_score=False,
        )
        search.fit(X_dev, y_dev)
        searches[model_name] = search

        bp = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
        best_params_scenario[model_name] = bp

        i = search.best_index_
        cv_rows.append({
            'scenario': scenario,
            'negative_ratio': ratio,
            'positive_prevalence_development': prevalence_dev,
            'model': model_name,
            'best_mean_auc_pr_cv': search.cv_results_['mean_test_auc_pr'][i],
            'std_auc_pr_cv': search.cv_results_['std_test_auc_pr'][i],
            'best_mean_auc_roc_cv': search.cv_results_['mean_test_auc_roc'][i],
            'std_auc_roc_cv': search.cv_results_['std_test_auc_roc'][i],
            'best_mean_f1_cv': search.cv_results_['mean_test_f1'][i],
            'std_f1_cv': search.cv_results_['std_test_f1'][i],
            'best_mean_accuracy_cv': search.cv_results_['mean_test_accuracy'][i],
            'std_accuracy_cv': search.cv_results_['std_test_accuracy'][i],
            'best_mean_balanced_accuracy_cv': search.cv_results_['mean_test_balanced_accuracy'][i],
            'std_balanced_accuracy_cv': search.cv_results_['std_test_balanced_accuracy'][i],
            'best_mean_precision_cv': search.cv_results_['mean_test_precision'][i],
            'best_mean_recall_cv': search.cv_results_['mean_test_recall'][i],
        })
        print(f"    AUC-PR CV={search.cv_results_['mean_test_auc_pr'][i]:.4f}, parametros={bp}")

    all_searches[scenario] = searches
    best_params_all[scenario] = best_params_scenario
    all_data[scenario] = {
        'train_df': train_df,
        'val_df': val_df,
        'dev_df': dev_df,
        'test_df': test_df,
        'X_test': X_test,
        'y_test': y_test,
        'prevalence_test': prevalence_test,
    }

    audit_rows.append({
        'experiment': 'P7-1_Evaluacion_Desbalance_Clases',
        'scenario': scenario,
        'negative_ratio': ratio,
        'threshold_coauthored_papers': 1,
        'top7_frozen_from_P5': True,
        'development_scope': 'train_plus_validation_2021_2022',
        'hyperparameter_selection_scope': f'RepeatedStratifiedKFold_{N_SPLITS}x{N_REPEATS}_development_only',
        'refit_metric': 'auc_pr',
        'preprocessing_scope': 'inside_each_CV_training_fold',
        'class_weighting': 'none',
        'test_used_for_selection': False,
        'run_final_test': RUN_FINAL_TEST,
        'n_train_original': len(train_df),
        'n_val_original': len(val_df),
        'n_development': len(dev_df),
        'n_test': len(test_df),
        'positive_prevalence_development': prevalence_dev,
        'positive_prevalence_test': prevalence_test,
        'seed': SEED,
    })

cv_summary = pd.DataFrame(cv_rows).sort_values(
    ['scenario', 'best_mean_auc_pr_cv'], ascending=[True, False]
).reset_index(drop=True)

cv_file = OUTPUT_DIR / 'P7-1_Desbalance_CV_model_metrics.csv'
params_file = OUTPUT_DIR / 'P7-1_Desbalance_best_params.json'
audit_file = OUTPUT_DIR / 'P7-1_Desbalance_audit.csv'

cv_summary.to_csv(cv_file, index=False)
with open(params_file, 'w', encoding='utf-8') as f:
    json.dump(best_params_all, f, indent=2, ensure_ascii=False)
pd.DataFrame(audit_rows).to_csv(audit_file, index=False)

print('\n' + '=' * 78)
print('RESUMEN CV POR ESCENARIO')
print('=' * 78)
print(cv_summary[[
    'scenario', 'model', 'best_mean_auc_pr_cv', 'std_auc_pr_cv',
    'best_mean_auc_roc_cv', 'best_mean_f1_cv', 'best_mean_balanced_accuracy_cv'
]].to_string(index=False))

print('\nArchivos de seleccion guardados:')
print(cv_file)
print(params_file)
print(audit_file)

# Evaluacion final. Solo activar despues de revisar y congelar la seleccion de CV.
if RUN_FINAL_TEST:
    test_rows = []

    for scenario, ratio in SCENARIOS.items():
        data = all_data[scenario]
        test_df = data['test_df']
        X_test = data['X_test']
        y_test = data['y_test']
        prevalence_test = data['prevalence_test']

        prediction_table = test_df[['source', 'target', TARGET]].copy() \
            if {'source', 'target'}.issubset(test_df.columns) \
            else pd.DataFrame({TARGET: y_test})

        for model_name, search in all_searches[scenario].items():
            final_model = search.best_estimator_
            y_pred = final_model.predict(X_test)
            y_score = decision_score(final_model, X_test)

            test_rows.append({
                'scenario': scenario,
                'negative_ratio': ratio,
                'positive_prevalence_test': prevalence_test,
                'model': model_name,
                'accuracy_test': accuracy_score(y_test, y_pred),
                'balanced_accuracy_test': balanced_accuracy_score(y_test, y_pred),
                'precision_test': precision_score(y_test, y_pred, zero_division=0),
                'recall_test': recall_score(y_test, y_pred, zero_division=0),
                'f1_test': f1_score(y_test, y_pred, zero_division=0),
                'auc_roc_test': roc_auc_score(y_test, y_score),
                'auc_pr_test': average_precision_score(y_test, y_score),
                'auc_pr_random_baseline': prevalence_test,
                'auc_pr_lift_over_baseline': average_precision_score(y_test, y_score) / prevalence_test if prevalence_test > 0 else np.nan,
            })
            prediction_table[f'{model_name}__pred'] = y_pred
            prediction_table[f'{model_name}__score'] = y_score

        safe = scenario.replace(':', 'to')
        prediction_table.to_csv(
            OUTPUT_DIR / f'P7-1_Desbalance_FINAL_test_predictions_{safe}.csv',
            index=False,
        )

    test_metrics = pd.DataFrame(test_rows).sort_values(
        ['scenario', 'auc_pr_test'], ascending=[True, False]
    ).reset_index(drop=True)
    test_file = OUTPUT_DIR / 'P7-1_Desbalance_FINAL_test_metrics.csv'
    test_metrics.to_csv(test_file, index=False)

    audit_df = pd.DataFrame(audit_rows)
    audit_df['run_final_test'] = True
    audit_df.to_csv(audit_file, index=False)

    print('\n' + '=' * 78)
    print('RESULTADOS FINALES EN TEST POR ESCENARIO')
    print('=' * 78)
    print(test_metrics[[
        'scenario', 'model', 'auc_pr_test', 'auc_pr_random_baseline',
        'auc_pr_lift_over_baseline', 'auc_roc_test', 'f1_test',
        'precision_test', 'recall_test', 'balanced_accuracy_test'
    ]].to_string(index=False))
    print('\nArchivo final:')
    print(test_file)
else:
    print('\nRUN_FINAL_TEST=False: los tests 2023-2024 NO fueron evaluados.')
    print('Revise primero CV y parametros; despues cambie unicamente RUN_FINAL_TEST=True.')


ESCENARIO 1:1  (1 positivo : 1 negativos)
train: n=27, clases={0: 13, 1: 14}
val: n=7, clases={0: 4, 1: 3}
desarrollo: n=34, clases={0: 17, 1: 17}, prevalencia positiva=0.5000
test reservado: n=112, clases={0: 56, 1: 56}, prevalencia positiva=0.5000
  Buscando hiperparametros: Decision Tree
    AUC-PR CV=0.7056, parametros={'criterion': 'gini', 'max_depth': 2, 'min_samples_leaf': 4}
  Buscando hiperparametros: SVM - Linear Kernel
    AUC-PR CV=0.8049, parametros={'C': 0.01, 'dual': False}
  Buscando hiperparametros: SVM - RBF Kernel
    AUC-PR CV=0.7818, parametros={'C': 1.0, 'gamma': 'scale'}
  Buscando hiperparametros: K-Neighbors
    AUC-PR CV=0.7755, parametros={'n_neighbors': 5, 'p': 1, 'weights': 'distance'}
  Buscando hiperparametros: Naive Bayes
    AUC-PR CV=0.7842, parametros={'var_smoothing': 1e-11}
  Buscando hiperparametros: Multi-Layer Perceptron
    AUC-PR CV=0.7887, parametros={'alpha': 0.0001, 'hidden_layer_sizes': (25,), 'learning_rate_init': 0.001}

ESCENARIO 1:5  (